# 03 — Silver Transformation
## Clean Silver → Analytically Enriched Silver

**Purpose:** Add derived columns that the business needs but the source never captured.  
No cleaning here — data enters this notebook already clean from Notebook 02.  
Every derived column is documented with its business purpose and formula.

**Input :** `sales_silver.clean_superstore`  
**Output:** `sales_silver.clean_superstore` (overwrite with enriched columns)  
**Layer  :** Silver (transform)


## 1. Load config

In [0]:
%run ./00_setup_and_config

In [0]:
log("INFO", "03_silver_transform", "Starting Silver transformation layer")


## 2. Read from silver clean table

In [0]:
df = spark.table(SILVER_TABLE)

log("INFO", "03_silver_transform", f"Read from Silver: {SILVER_TABLE}")
print(f"  Rows     : {df.count():,}")
print(f"  Columns  : {len(df.columns)}")

# Quick sanity check — confirm key columns exist before transforming
required_cols = [
    "order_date", "ship_date", "sales",
    "profit", "discount", "quantity"
]

missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise Exception(f"Required columns missing from Silver: {missing}")

print(f"  ✅ All required columns present")


## 3. Extract data parts for time-series analysis

In [0]:
from pyspark.sql import functions as F

df = df.withColumns({
    "order_year"      : F.year("order_date"),
    "order_month"     : F.month("order_date"),
    "order_quarter"   : F.quarter("order_date"),
    "order_day_of_week" : F.dayofweek("order_date"),  # 1=Sunday, 7=Saturday
    "order_yearmonth" : F.date_format("order_date", "yyyy-MM"),
})

# Verify date parts look correct
print("=== DATE DERIVATIONS SAMPLE ===\n")
df.select(
    "order_date", "order_year", "order_month",
    "order_quarter", "order_yearmonth"
).show(5, truncate=False)

log("INFO", "03_silver_transform", "Date columns derived: year, month, quarter, yearmonth")


## 4. calculate shipping duration in days

In [0]:
df = df.withColumn(
    "shipping_days",
    F.datediff(F.col("ship_date"), F.col("order_date"))
)

# Categorise shipping speed based on days
# Same Day = 0 days, First Class = 1-2, Second Class = 3-4, Standard = 5+
df = df.withColumn(
    "shipping_speed",
    F.when(F.col("shipping_days") == 0, "Same Day")
     .when(F.col("shipping_days") <= 2, "Fast")
     .when(F.col("shipping_days") <= 4, "Standard")
     .otherwise("Slow")
)

print("=== SHIPPING DAYS DISTRIBUTION ===\n")
df.groupBy("shipping_speed") \
  .count() \
  .orderBy("shipping_speed") \
  .show()

log("INFO", "03_silver_transform", "shipping_days and shipping_speed derived")


## 5. Derive profit-related columns

In [0]:
# profit_margin = profit / sales
#   Expressed as a ratio (0.25 = 25% margin)

# is_profitable = profit > 0
#   Boolean flag. Lets analysts quickly filter profitable vs loss-making.

df = df.withColumns({
    "profit_margin" : F.when(
                          F.col("sales") != 0,
                          F.round(F.col("profit") / F.col("sales"), 4)
                      ).otherwise(0.0),

    "is_profitable" : F.col("profit") > 0,
})

# Sanity check on profit margin range
print("=== PROFIT MARGIN STATISTICS ===\n")
df.select(
    F.round(F.min("profit_margin"), 4).alias("min_margin"),
    F.round(F.avg("profit_margin"), 4).alias("avg_margin"),
    F.round(F.max("profit_margin"), 4).alias("max_margin"),
).show()

print("=== PROFITABLE vs LOSS-MAKING ===\n")
df.groupBy("is_profitable") \
  .count() \
  .withColumnRenamed("count", "transaction_count") \
  .show()

log("INFO", "03_silver_transform", "profit_margin and is_profitable derived")


## 6. Sales and discount banding (bucketing)

In [0]:
# Sales bands — based on order of magnitude:
#   Low       : < $100      (small office supplies, accessories)
#   Medium    : $100-$500   (mid-range products)
#   High      : $500-$2000  (furniture, high-end tech)
#   Very High : > $2000     (enterprise purchases)

df = df.withColumn(
    "sales_band",
    F.when(F.col("sales") < 100,   "Low")
     .when(F.col("sales") < 500,   "Medium")
     .when(F.col("sales") < 2000,  "High")
     .otherwise("Very High")
)

df = df.withColumn(
    "discount_band",
    F.when(F.col("discount") == 0,    "No Discount")
     .when(F.col("discount") <= 0.2,  "Low")
     .when(F.col("discount") <= 0.4,  "Medium")
     .otherwise("High")
)

print("=== SALES BAND DISTRIBUTION ===\n")
df.groupBy("sales_band") \
  .count() \
  .orderBy("sales_band") \
  .show()

print("=== DISCOUNT BAND DISTRIBUTION ===\n")
df.groupBy("discount_band") \
  .count() \
  .orderBy("discount_band") \
  .show()

log("INFO", "03_silver_transform", "sales_band and discount_band derived")


## 7. Gross revenue and loss columns

In [0]:

# The source `sales` column already represents revenue after discount.
# We derive two additional revenue views:
#
#   unit_price = sales / (quantity * (1 - discount))
#     The original price before discount, per unit.
#     Useful for product pricing analysis.
#
#   absolute_discount_amount = unit_price * quantity * discount
#     The actual dollar value of discount given. More meaningful than
#     the discount ratio alone for financial reporting.

df = df.withColumns({
    "unit_price" : F.when(
                       (F.col("quantity") > 0) & (F.col("discount") < 1),
                       F.round(
                           F.col("sales") / (
                               F.col("quantity") * (1 - F.col("discount"))
                           ), 2
                       )
                   ).otherwise(F.col("sales")),

    "discount_amount" : F.round(
                            F.col("sales") * F.col("discount"), 2
                        ),
})

print("=== REVENUE DERIVATIONS SAMPLE ===\n")
df.select(
    "sales", "quantity", "discount",
    "unit_price", "discount_amount", "profit_margin"
).show(5, truncate=False)

log("INFO", "03_silver_transform", "unit_price and discount_amount derived")


## 8. Review the full enriched schema before writing

In [0]:
# Cell 9: Review the full enriched schema before writing
#
# At this point Silver has grown from 21 source columns to a much
# richer analytical dataset. Document the final shape.

print("=== ENRICHED SILVER SCHEMA ===\n")

source_cols  = []
derived_cols = []
meta_cols    = []

derived_names = [
    "order_year", "order_month", "order_quarter",
    "order_day_of_week", "order_yearmonth",
    "shipping_days", "shipping_speed",
    "profit_margin", "is_profitable",
    "sales_band", "discount_band",
    "unit_price", "discount_amount",
]

for field in df.schema.fields:
    if field.name.startswith("_"):
        meta_cols.append(field)
    elif field.name in derived_names:
        derived_cols.append(field)
    else:
        source_cols.append(field)

print(f"  Source columns  ({len(source_cols):02d}) :", [f.name for f in source_cols])
print(f"  Derived columns ({len(derived_cols):02d}) :", [f.name for f in derived_cols])
print(f"  Metadata columns({len(meta_cols):02d}) :", [f.name for f in meta_cols])
print(f"\n  Total columns   : {len(df.columns)}")


## 9. Overwrite Silver table with enriched version

In [0]:
# We overwrite the same Silver table (clean_superstore) rather than
# creating a new one. This keeps the layer boundary clean:
#   Bronze  = raw_superstore
#   Silver  = clean_superstore  (clean + transformed — one table)
#   Gold    = fact and dimension tables

(
    df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_TABLE)
)

log("INFO", "03_silver_transform", f"Enriched Silver written: {SILVER_TABLE}")


## 10. Verify the enriched Silver table

In [0]:
df_verify = spark.table(SILVER_TABLE)

print("=== TRANSFORMATION VERIFICATION ===\n")
print(f"  Rows in enriched Silver : {df_verify.count():,}")
print(f"  Columns                 : {len(df_verify.columns)}")

# Spot-check derived columns are populated (not all null)
derived_check = df_verify.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in derived_names
    if c in df_verify.columns
]).collect()[0]

print("\n  Null check on derived columns:")
all_good = True
for col_name, null_count in zip(derived_names, derived_check):
    if col_name not in df_verify.columns:
        continue
    flag = "✅" if null_count == 0 else f"⚠️  {null_count} nulls"
    print(f"    {col_name:<30} : {flag}")
    if null_count > 0:
        all_good = False

if all_good:
    print("\n  ✅ All derived columns populated correctly")

log("INFO", "03_silver_transform", "Transformation verification complete")


## 12. Notebook completion summary

In [0]:
print("=" * 55)
print("  SILVER TRANSFORMATION — COMPLETE")
print("=" * 55)
print(f"  Table    : {SILVER_TABLE}")
print(f"  Rows     : {df_verify.count():,}")
print(f"  Derived  : {len(derived_names)} new analytical columns")
print(f"  Ready for: Gold layer (fact + dimension tables)")
print("=" * 55)

log("INFO", "03_silver_transform", "Notebook 03 complete ✅")